## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### PART A: Divide our documents into chunks

In [47]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [48]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-proj-


In [49]:
import glob
from langchain_community.document_loaders import PyPDFLoader

# 1. Locate the files
knowledge_base_path = "Marcela_Doc/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")


Found 1 files in the knowledge base
Total characters in knowledge base: 27,372


In [50]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 6,133


In [51]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("Marcela_Doc/")

documents = []
for folder in folders:
    # 🔴 COMPROBACIÓN CRÍTICA: Solo procesar si realmente es una carpeta
    if os.path.isdir(folder):
        doc_type = os.path.basename(folder)
        loader = DirectoryLoader(folder, glob="*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
        folder_docs = loader.load()
        for doc in folder_docs:
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 1 documents


In [52]:
documents[0]

Document(metadata={'source': 'Marcela_Doc\\cuento.md', 'doc_type': ''}, page_content='El científico y su asistente.\n\nMi nombre es Rolando, un niño con muchas inquietudes y mucha curiosidad; junto con mi asistente-mascota, El Canicas, trabajamos a marchas forzadas en un proyecto que tiene un sentido muy especial para nosotros.\nPrimero debo aclarar que El Canicas tal vez no sea un asistente normal; para la mayoría de las personas es solo un perro de compañía, para mi es parte de mi equipo y un integrante muy importante.\n\n\nAún recuerdo el día que llegó a casa, mi papá lo trajo de manera inesperada y desde ese día nos hemos vuelto inseparables. Ha sido un buen compañero de juegos y de vida científica. Siempre está de buen humor y es demasiado emotivo; le pone mucha alegría a mis días. Detrás de esas orejas caídas y mechones de pelo café esta mi mejor compañero de juegos y de investigación. Me pregunto si mi papá vio en él lo mismo que yo al momento de traerlo del refugio, me gusta pe

In [53]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 43 chunks
First chunk:

page_content='El científico y su asistente.

Mi nombre es Rolando, un niño con muchas inquietudes y mucha curiosidad; junto con mi asistente-mascota, El Canicas, trabajamos a marchas forzadas en un proyecto que tiene un sentido muy especial para nosotros.
Primero debo aclarar que El Canicas tal vez no sea un asistente normal; para la mayoría de las personas es solo un perro de compañía, para mi es parte de mi equipo y un integrante muy importante.' metadata={'source': 'Marcela_Doc\\cuento.md', 'doc_type': ''}


In [54]:
chunks[42]

Document(metadata={'source': 'Marcela_Doc\\cuento.md', 'doc_type': ''}, page_content='Aunque las pócimas no funcionen la abuela suele recomendarme que continúe con mis experimentos científicos y quizás algún día logre descubrir la cura para alguna enfermedad.\nYo también creo que tengo madera de científico y sé que con más años de estudio y práctica algún día lograré formar parte de un gran equipo de investigadores que aporten soluciones a las enfermedades de las personas. Tal vez hasta llegue a ganar un premio Nobel con mis investigaciones y mi\n \ntrabajo llegue a muchas personas para solucionar algunas de sus necesidades.\n\n\nTal vez no involucren plantas mágicas o brebajes que contengan imaginación como ahora, pero estas experiencias sin duda me ayudan a mejorar mi método pues la creatividad es algo fundamental en un científico, además de saber observar y ser paciente cuando se está en el laboratorio.')

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

In [55]:
# Pick an embedding model

#embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 43 documents


In [56]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 43 vectors with 3,072 dimensions in the vector store


### Part C: Visualize!

In [44]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

ValueError: '' is not in list

In [45]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

NameError: name 'colors' is not defined

In [30]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

NameError: name 'vectors' is not defined